In [1]:
import optuna

In [2]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
import numpy as np

cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

df.fillna(df.mean(), inplace=True)

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [5]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [7]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-06-10 11:06:25,415] A new study created in memory with name: no-name-79e9f917-f05b-4385-ba74-294f766a707d
[I 2025-06-10 11:06:26,308] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 89, 'max_depth': 15}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-10 11:06:28,148] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 136, 'max_depth': 6}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-10 11:06:30,173] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 141, 'max_depth': 12}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-06-10 11:06:30,974] Trial 3 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 90, 'max_depth': 3}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-06-10 11:06:31,724] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 77, 'max_depth': 5}. Best is trial 2 with value: 0.7728119180

In [8]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 107, 'max_depth': 15}


In [9]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.76


# Sampler in Optuna

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [11]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-06-10 11:07:40,502] A new study created in memory with name: no-name-8ae86fbf-4417-437b-a129-f017caf927e8
[I 2025-06-10 11:07:42,433] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 138, 'max_depth': 9}. Best is trial 0 with value: 0.7635009310986964.
[I 2025-06-10 11:07:43,724] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 95, 'max_depth': 6}. Best is trial 0 with value: 0.7635009310986964.
[I 2025-06-10 11:07:44,848] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 111, 'max_depth': 17}. Best is trial 2 with value: 0.7746741154562384.
[I 2025-06-10 11:07:46,072] Trial 3 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 116, 'max_depth': 15}. Best is trial 3 with value: 0.7765363128491621.
[I 2025-06-10 11:07:47,163] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 3 with value: 0.77653631

In [12]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 123, 'max_depth': 17}


In [13]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


In [14]:
## Grid search

search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [15]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-06-10 11:08:49,314] A new study created in memory with name: no-name-15c0fd49-e9a5-4b66-a8a0-801baf95d814
[I 2025-06-10 11:08:50,288] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-10 11:08:51,873] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-06-10 11:08:52,579] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-06-10 11:08:53,670] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-06-10 11:08:54,828] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [16]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [17]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


# Optuna Vizulizations

In [18]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [19]:
# 1. Optimization History
plot_optimization_history(study).show()

In [20]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [21]:
# 3. Slice Plot
plot_slice(study).show()

In [22]:
# 4. Contour Plot
plot_contour(study).show()

In [23]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

# Optimizing multiple model

In [24]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [25]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [26]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-06-10 11:11:08,135] A new study created in memory with name: no-name-63117afa-2792-42e6-aee4-e4061f341642
[I 2025-06-10 11:11:11,764] Trial 0 finished with value: 0.7597765363128491 and parameters: {'classifier': 'RandomForest', 'n_estimators': 191, 'max_depth': 7, 'min_samples_split': 4, 'min_samples_leaf': 10, 'bootstrap': True}. Best is trial 0 with value: 0.7597765363128491.
[I 2025-06-10 11:11:15,704] Trial 1 finished with value: 0.7355679702048418 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 82, 'learning_rate': 0.09480523442065009, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.7597765363128491.
[I 2025-06-10 11:11:15,784] Trial 2 finished with value: 0.7243947858472999 and parameters: {'classifier': 'SVM', 'C': 1.3333157230142283, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 0 with value: 0.7597765363128491.
[I 2025-06-10 11:11:15,836] Trial 3 finished with value: 0.7299813780260708 and paramete

In [27]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.12794321169134254, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [28]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.759777,2025-06-10 11:11:08.138171,2025-06-10 11:11:11.763769,0 days 00:00:03.625598,NaN,True,RandomForest,NaN,NaN,NaN,7.0,10.0,4.0,191.0,COMPLETE
1,1,0.735568,2025-06-10 11:11:11.767158,2025-06-10 11:11:15.703966,0 days 00:00:03.936808,NaN,NaN,GradientBoosting,NaN,NaN,0.094805,17.0,5.0,4.0,82.0,COMPLETE
2,2,0.724395,2025-06-10 11:11:15.705860,2025-06-10 11:11:15.784178,0 days 00:00:00.078318,1.333316,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.729981,2025-06-10 11:11:15.786006,2025-06-10 11:11:15.836466,0 days 00:00:00.050460,0.906965,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.763501,2025-06-10 11:11:15.838518,2025-06-10 11:11:16.719739,0 days 00:00:00.881221,NaN,True,RandomForest,NaN,NaN,NaN,11.0,3.0,2.0,94.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.785847,2025-06-10 11:12:56.753143,2025-06-10 11:12:57.043929,0 days 00:00:00.290786,42.415068,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2025-06-10 11:12:57.045667,2025-06-10 11:12:57.518041,0 days 00:00:00.472374,80.148067,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.785847,2025-06-10 11:12:57.519527,2025-06-10 11:12:58.013882,0 days 00:00:00.494355,59.880997,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.785847,2025-06-10 11:12:58.016588,2025-06-10 11:12:58.094505,0 days 00:00:00.077917,0.192715,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [29]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 71
RandomForest        20
GradientBoosting     9
Name: count, dtype: int64

In [30]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.740327
RandomForest        0.767039
SVM                 0.769691
Name: value, dtype: float64

In [31]:
# 1. Optimization History
plot_optimization_history(study).show()

In [32]:
# 3. Slice Plot
plot_slice(study).show()

In [33]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

In [36]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2025-06-10 11:13:35,582] A new study created in memory with name: no-name-249b5294-83dc-450b-b076-cba8535119fa


[0]	train-mlogloss:0.88018	eval-mlogloss:0.87211
[1]	train-mlogloss:0.79557	eval-mlogloss:0.77344
[2]	train-mlogloss:0.67049	eval-mlogloss:0.64056
[3]	train-mlogloss:0.56019	eval-mlogloss:0.51874
[4]	train-mlogloss:0.47511	eval-mlogloss:0.43120
[5]	train-mlogloss:0.40769	eval-mlogloss:0.36461
[6]	train-mlogloss:0.34896	eval-mlogloss:0.29890
[7]	train-mlogloss:0.30724	eval-mlogloss:0.25900
[8]	train-mlogloss:0.27603	eval-mlogloss:0.22307
[9]	train-mlogloss:0.24606	eval-mlogloss:0.19400
[10]	train-mlogloss:0.22281	eval-mlogloss:0.16589
[11]	train-mlogloss:0.20493	eval-mlogloss:0.14525
[12]	train-mlogloss:0.19594	eval-mlogloss:0.13811
[13]	train-mlogloss:0.18420	eval-mlogloss:0.12579
[14]	train-mlogloss:0.18300	eval-mlogloss:0.12585
[15]	train-mlogloss:0.17534	eval-mlogloss:0.11688
[16]	train-mlogloss:0.17335	eval-mlogloss:0.11420
[17]	train-mlogloss:0.16857	eval-mlogloss:0.10717
[18]	train-mlogloss:0.16783	eval-mlogloss:0.10914
[19]	train-mlogloss:0.16225	eval-mlogloss:0.10307
[20]	train

[I 2025-06-10 11:13:36,613] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.00013695093332539675, 'alpha': 0.0008298234829026117, 'eta': 0.1824635846276565, 'gamma': 0.0006890317133960559, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.4219159808455654, 'colsample_bytree': 0.56114666874105}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.77069	eval-mlogloss:0.75920
[1]	train-mlogloss:0.57657	eval-mlogloss:0.55198
[2]	train-mlogloss:0.44118	eval-mlogloss:0.40944
[3]	train-mlogloss:0.34797	eval-mlogloss:0.31039
[4]	train-mlogloss:0.28251	eval-mlogloss:0.23795
[5]	train-mlogloss:0.23371	eval-mlogloss:0.18347
[6]	train-mlogloss:0.20098	eval-mlogloss:0.14540
[7]	train-mlogloss:0.17095	eval-mlogloss:0.11566
[8]	train-mlogloss:0.14930	eval-mlogloss:0.09748
[9]	train-mlogloss:0.13029	eval-mlogloss:0.08626
[10]	train-mlogloss:0.11618	eval-mlogloss:0.07356
[11]	train-mlogloss:0.11030	eval-mlogloss:0.06852
[12]	train-mlogloss:0.10519	eval-mlogloss:0.06406
[13]	train-mlogloss:0.10259	eval-mlogloss:0.06111
[14]	train-mlogloss:0.10175	eval-mlogloss:0.06230
[15]	train-mlogloss:0.10172	eval-mlogloss:0.06228
[16]	train-mlogloss:0.10024	eval-mlogloss:0.05922
[17]	train-mlogloss:0.09694	eval-mlogloss:0.05916
[18]	train-mlogloss:0.09623	eval-mlogloss:0.05712
[19]	train-mlogloss:0.09492	eval-mlogloss:0.05567
[20]	train

[I 2025-06-10 11:13:39,096] Trial 1 finished with value: 1.0 and parameters: {'lambda': 0.001062152070410872, 'alpha': 0.16013739547671027, 'eta': 0.27700056276112417, 'gamma': 0.3799195409952126, 'max_depth': 4, 'min_child_weight': 3, 'subsample': 0.902606582072063, 'colsample_bytree': 0.9665681373104906}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.03591	eval-mlogloss:1.03259
[1]	train-mlogloss:0.99706	eval-mlogloss:0.99965
[2]	train-mlogloss:0.94607	eval-mlogloss:0.94582
[3]	train-mlogloss:0.89410	eval-mlogloss:0.89185
[4]	train-mlogloss:0.84733	eval-mlogloss:0.84325


[I 2025-06-10 11:13:53,229] Trial 48 pruned. Trial was pruned at iteration 4.


[0]	train-mlogloss:1.08528	eval-mlogloss:1.08477
[1]	train-mlogloss:1.07223	eval-mlogloss:1.07086
[2]	train-mlogloss:1.05885	eval-mlogloss:1.05686
[3]	train-mlogloss:1.04558	eval-mlogloss:1.04321
[4]	train-mlogloss:1.03313	eval-mlogloss:1.03041
[5]	train-mlogloss:1.02039	eval-mlogloss:1.01690
[6]	train-mlogloss:1.00851	eval-mlogloss:1.00437
[7]	train-mlogloss:0.99700	eval-mlogloss:0.99174
[8]	train-mlogloss:0.98526	eval-mlogloss:0.97914
[9]	train-mlogloss:0.97369	eval-mlogloss:0.96668
[10]	train-mlogloss:0.96233	eval-mlogloss:0.95499
[11]	train-mlogloss:0.95119	eval-mlogloss:0.94319
[12]	train-mlogloss:0.94037	eval-mlogloss:0.93186
[13]	train-mlogloss:0.92945	eval-mlogloss:0.92060
[14]	train-mlogloss:0.91901	eval-mlogloss:0.90936
[15]	train-mlogloss:0.90851	eval-mlogloss:0.89813
[16]	train-mlogloss:0.89814	eval-mlogloss:0.88714
[17]	train-mlogloss:0.88835	eval-mlogloss:0.87669
[18]	train-mlogloss:0.87844	eval-mlogloss:0.86631
[19]	train-mlogloss:0.86856	eval-mlogloss:0.85629
[20]	train

[I 2025-06-10 11:13:53,742] Trial 49 pruned. Trial was pruned at iteration 64.


Best trial: {'lambda': 0.00013695093332539675, 'alpha': 0.0008298234829026117, 'eta': 0.1824635846276565, 'gamma': 0.0006890317133960559, 'max_depth': 7, 'min_child_weight': 3, 'subsample': 0.4219159808455654, 'colsample_bytree': 0.56114666874105}
Best accuracy: 1.0


In [35]:
! pip install optuna-integration[xgboost]

Defaulting to user installation because normal site-packages is not writeable


In [37]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()